In [ ]:
%pip install -q vllm sentence-transformers tqdm implicit

In [ ]:
import os

os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'

import gc
import json
import re
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.neighbors import NearestNeighbors
from sentence_transformers import SentenceTransformer
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('../../data/children_products')
CLEANED_CSV = DATA_DIR / 'clildren_product_cleaned.csv'
RAW_CSV = DATA_DIR / '!03&04_17_VSE.csv'

MIN_INTERACTIONS = 5
TRAIN_SPLIT = 0.8

N_EVAL_USERS = 1000
K_CANDIDATES = 50
FINAL_K = 20
MAX_HISTORY_ITEMS = 20

EMBED_MODEL = 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2'
MODEL_CHOICE = 'qwen7b'  # 'qwen3b' | 'qwen7b' | 'mistral7b' | 'vikhr12b' | 'qwen14b' | 'qwen32b_awq'
MODEL_REGISTRY = {
    'qwen3b':      ('Qwen/Qwen2.5-3B-Instruct',                          'qwen3b',      None,  6,  'qwen3b'),
    'qwen7b':      ('Qwen/Qwen2.5-7B-Instruct',                          'qwen7b',      None,  14, 'qwen7b'),
    'mistral7b':   ('mistralai/Mistral-7B-Instruct-v0.3',                'mistral7b',   None,  14, 'mistral7b'),
    'vikhr12b':    ('Vikhrmodels/Vikhr-Nemo-12B-Instruct-R-21-09-24',    'vikhr12b',    None,  24, 'vikhr12b'),
    'qwen14b':     ('Qwen/Qwen2.5-14B-Instruct',                         'qwen14b',     None,  28, 'qwen14b'),
    'qwen32b_awq': ('Qwen/Qwen2.5-32B-Instruct-AWQ',                     'qwen32b_awq', 'awq', 17, 'qwen32b_awq'),
}
LLM_MODEL, MODEL_SUFFIX, LLM_QUANT, _vram_est, _note = MODEL_REGISTRY[MODEL_CHOICE]
print(f'Using LLM: {LLM_MODEL}  (~{_vram_est} GB, {_note})')

SKU_EMB_CACHE = DATA_DIR / 'sku_embeddings.npy'
LLM_RERANK_V2_CACHE = DATA_DIR / f'llm_rerank_v2_cache_vllm_{MODEL_SUFFIX}.json'
CANDIDATES_CACHE = DATA_DIR / 'rerank_base_candidates.npz'
NCF_CHECKPOINT = DATA_DIR / 'ncf_state.pt'

RRF_K = 60
ALPHA_SWEEP = [0.0, 0.1, 0.3, 0.5, 1.0]

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)


In [ ]:
df = pd.read_csv(CLEANED_CSV)
print(f'Исходный датасет: {df.shape}')

raw = pd.read_csv(
    RAW_CSV,
    sep=';', encoding='cp1251',
    usecols=['ID_SKU', 'Номенклатура', 'Группа2', 'Группа3', 'Тип'],
    dtype=str, low_memory=False
)
raw = raw.dropna(subset=['ID_SKU']).drop_duplicates('ID_SKU')

df = df.merge(raw[['ID_SKU', 'Номенклатура']], on='ID_SKU', how='left')
fallback = df['Группа3'].fillna('') + ' / ' + df['Тип'].fillna('')
df['Номенклатура'] = df['Номенклатура'].fillna(fallback)

df_filtered = df[(df['Статус'] == 'Доставлен') & (df['Отменено'] == 'Нет')].copy()
df_filtered = df_filtered.dropna(subset=['Телефон_new', 'ID_SKU', 'Дата'])
df_filtered['Дата'] = pd.to_datetime(df_filtered['Дата'], errors='coerce')
df_filtered = df_filtered.dropna(subset=['Дата'])

for _ in range(5):
    uc = df_filtered.groupby('Телефон_new').size()
    ic = df_filtered.groupby('ID_SKU').size()
    active_users = uc[uc >= MIN_INTERACTIONS].index
    active_items = ic[ic >= MIN_INTERACTIONS].index
    before = len(df_filtered)
    df_filtered = df_filtered[
        df_filtered['Телефон_new'].isin(active_users) &
        df_filtered['ID_SKU'].isin(active_items)
    ]
    if len(df_filtered) == before:
        break

user_enc, item_enc = LabelEncoder(), LabelEncoder()
df_filtered['user_id'] = user_enc.fit_transform(df_filtered['Телефон_new'])
df_filtered['item_id'] = item_enc.fit_transform(df_filtered['ID_SKU'])
n_users, n_items = len(user_enc.classes_), len(item_enc.classes_)
print(f'Пользователей: {n_users:,}  Товаров: {n_items:,}  Взаимодействий: {len(df_filtered):,}')

df_filtered = df_filtered.sort_values('Дата')
split_date = df_filtered['Дата'].quantile(TRAIN_SPLIT)
train_data = df_filtered[df_filtered['Дата'] < split_date].copy()
test_data = df_filtered[df_filtered['Дата'] >= split_date].copy()
print(f'Split date: {split_date}')
print(f'Train: {len(train_data):,}  Test: {len(test_data):,}')

train_int = train_data.groupby(['user_id', 'item_id']).size().reset_index(name='count')
test_int = test_data.groupby(['user_id', 'item_id']).size().reset_index(name='count')
train_matrix = csr_matrix(
    (train_int['count'].values, (train_int['user_id'].values, train_int['item_id'].values)),
    shape=(n_users, n_items)
)

test_user_items = test_int.groupby('user_id')['item_id'].apply(list).to_dict()
train_user_set = set(train_int['user_id'].unique())
warm_eval_users = [u for u in test_user_items if u in train_user_set]

rng = np.random.default_rng(RANDOM_SEED)
sample_users = rng.choice(warm_eval_users, size=min(N_EVAL_USERS, len(warm_eval_users)), replace=False)
sample_users = sorted(map(int, sample_users))
print(f'Подвыборка для оценки: {len(sample_users)} пользователей')

In [ ]:
user_features = train_data.groupby('user_id').agg(
    user_n_purchases=('user_id', 'size'),
    user_n_unique_items=('item_id', 'nunique'),
    user_avg_price=('Цена', 'mean'),
).reset_index()

user_geo = train_data.groupby('user_id')['Гео'].agg(
    lambda x: x.mode().iloc[0] if len(x.mode()) else 'Неизвестно'
).reset_index()
user_geo.columns = ['user_id', 'user_geo']

user_delivery = train_data.groupby('user_id')['МетодДоставки_Групп'].agg(
    lambda x: x.mode().iloc[0] if len(x.mode()) else 'Неизвестно'
).reset_index()
user_delivery.columns = ['user_id', 'user_delivery_method']

user_features = user_features.merge(user_geo, on='user_id').merge(user_delivery, on='user_id')

user_top_cats = (
    train_data.groupby(['user_id', 'Группа2']).size()
    .reset_index(name='cnt')
    .sort_values(['user_id', 'cnt'], ascending=[True, False])
    .groupby('user_id')['Группа2']
    .apply(lambda s: list(s.head(3)))
    .reset_index()
)
user_top_cats.columns = ['user_id', 'user_top_cats']
user_features = user_features.merge(user_top_cats, on='user_id', how='left')
user_features['user_top_cats'] = user_features['user_top_cats'].apply(
    lambda x: x if isinstance(x, list) else []
)

user_feat_dict = user_features.set_index('user_id').to_dict('index')
print(f'User features: {len(user_feat_dict):,} пользователей')
print('  пример:', user_feat_dict[sample_users[0]])


In [ ]:
item_features = train_data.groupby('item_id').agg(
    item_avg_price=('Цена', 'mean'),
    item_n_purchases=('item_id', 'size'),
).reset_index()

# Группа2, Группа3, Тип — модальные значения
for col in ['Группа2', 'Группа3', 'Тип']:
    if col in train_data.columns:
        modal = train_data.groupby('item_id')[col].agg(
            lambda x: x.mode().iloc[0] if len(x.mode()) else 'Неизвестно'
        ).reset_index()
        modal.columns = ['item_id', col]
        item_features = item_features.merge(modal, on='item_id', how='left')

item_feat_dict = item_features.set_index('item_id').to_dict('index')
print(f'Item features: {len(item_feat_dict):,} товаров')
print('  пример:', item_feat_dict[next(iter(item_feat_dict))])


In [ ]:
item_name_map = (
    df_filtered.groupby('item_id')['Номенклатура']
    .agg(lambda s: s.mode().iloc[0] if len(s.mode()) else s.iloc[0])
)
item_names = [item_name_map.loc[i] for i in range(n_items)]
print(f'Номенклатур: {len(item_names):,}')
print(f'  примеры: {item_names[:3]}')


In [ ]:
train_sorted = train_data.sort_values('Дата')
user_history_cache = {}
for uid, g in train_sorted.groupby('user_id'):
    user_history_cache[int(uid)] = list(zip(g['Дата'].dt.date.astype(str), g['Номенклатура']))
user_train_items_cache = {
    int(uid): set(g['item_id'].tolist())
    for uid, g in train_sorted.groupby('user_id')
}

def build_user_history(user_id: int, max_items: int = MAX_HISTORY_ITEMS) -> str:
    items = user_history_cache.get(user_id, [])
    tail = items[-max_items:]
    return '\n'.join(f'- {date}: {name}' for date, name in tail)


In [ ]:
try:
    from implicit.gpu.als import AlternatingLeastSquares
except ImportError:
    from implicit.als import AlternatingLeastSquares

als = AlternatingLeastSquares(
    factors=34, regularization=0.2426, iterations=15,
    calculate_training_loss=True, random_state=RANDOM_SEED
)
als.fit(train_matrix)

als_candidates = {}
for uid in tqdm(sample_users, desc='ALS top-50'):
    ids, scores = als.recommend(
        uid, train_matrix[uid],
        N=K_CANDIDATES, filter_already_liked_items=True
    )
    als_candidates[int(uid)] = list(zip([int(x) for x in ids], [float(s) for s in scores]))

print(f'ALS candidates готовы для {len(als_candidates)} пользователей')


In [ ]:
df_for_neg = pd.read_csv(CLEANED_CSV)
df_hard = df_for_neg[
    df_for_neg['Статус'].isin(['Отменен', 'Возврат']) | (df_for_neg['Отменено'] == 'Да')
].copy()
df_hard = df_hard.dropna(subset=['Телефон_new', 'ID_SKU'])

phone_to_uid = dict(zip(df_filtered['Телефон_new'], df_filtered['user_id']))
sku_to_iid = dict(zip(df_filtered['ID_SKU'], df_filtered['item_id']))
df_hard['user_id'] = df_hard['Телефон_new'].map(phone_to_uid)
df_hard['item_id'] = df_hard['ID_SKU'].map(sku_to_iid)
df_hard = df_hard.dropna(subset=['user_id', 'item_id'])
df_hard['user_id'] = df_hard['user_id'].astype(int)
df_hard['item_id'] = df_hard['item_id'].astype(int)

positives = train_int[['user_id', 'item_id']].copy()
positives['label'] = 1.0

hard_negs = df_hard[['user_id', 'item_id']].drop_duplicates().copy()
hard_negs['label'] = 0.0

n_easy = len(positives) * 8
positive_pairs = set(zip(positives['user_id'].astype(int), positives['item_id'].astype(int)))
rng_neg = np.random.default_rng(RANDOM_SEED)
easy_users, easy_items = [], []
while len(easy_users) < n_easy:
    u_chunk = rng_neg.integers(0, n_users, size=n_easy * 2)
    i_chunk = rng_neg.integers(0, n_items, size=n_easy * 2)
    for u, i in zip(u_chunk.tolist(), i_chunk.tolist()):
        if (u, i) not in positive_pairs:
            easy_users.append(u)
            easy_items.append(i)
            if len(easy_users) >= n_easy:
                break

easy_negs = pd.DataFrame({'user_id': easy_users, 'item_id': easy_items, 'label': 0.0})
train_samples = pd.concat([
    positives[['user_id', 'item_id', 'label']],
    hard_negs[['user_id', 'item_id', 'label']],
    easy_negs[['user_id', 'item_id', 'label']]
], ignore_index=True).sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)
print(f'NCF train: total={len(train_samples):,}')


class NCF(nn.Module):
    def __init__(self, num_users, num_items, emb_dim=62, dropout=0.4935050):
        super().__init__()
        self.user_embedding_gmf = nn.Embedding(num_users, emb_dim)
        self.item_embedding_gmf = nn.Embedding(num_items, emb_dim)
        self.user_embedding_mlp = nn.Embedding(num_users, emb_dim)
        self.item_embedding_mlp = nn.Embedding(num_items, emb_dim)
        self.mlp = nn.Sequential(
            nn.Linear(emb_dim * 2, 128),
            nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(dropout),
        )
        self.fc_out = nn.Linear(emb_dim + 64, 1)

    def forward(self, user_ids, item_ids):
        u_gmf = self.user_embedding_gmf(user_ids)
        v_gmf = self.item_embedding_gmf(item_ids)
        gmf_out = u_gmf * v_gmf
        u_mlp = self.user_embedding_mlp(user_ids)
        v_mlp = self.item_embedding_mlp(item_ids)
        mlp_out = self.mlp(torch.cat([u_mlp, v_mlp], dim=-1))
        x = torch.cat([gmf_out, mlp_out], dim=-1)
        return torch.sigmoid(self.fc_out(x).squeeze(-1))


class NCFDataset(Dataset):
    def __init__(self, user_ids, item_ids, labels):
        self.user_ids = torch.LongTensor(user_ids)
        self.item_ids = torch.LongTensor(item_ids)
        self.labels = torch.FloatTensor(labels)

    def __len__(self):
        return len(self.user_ids)

    def __getitem__(self, idx):
        return self.user_ids[idx], self.item_ids[idx], self.labels[idx]


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
ncf_model = NCF(n_users, n_items, emb_dim=62, dropout=0.4935050).to(device)

if NCF_CHECKPOINT.exists():
    try:
        ncf_model.load_state_dict(torch.load(NCF_CHECKPOINT, map_location=device))
        print(f'NCF загружен из {NCF_CHECKPOINT}')
    except RuntimeError as e:
        print(f'Чекпойнт несовместим с архитектурой, переобучаем: {e}')
        NCF_CHECKPOINT.unlink()
else:
    train_ds = NCFDataset(
        train_samples['user_id'].values.astype(np.int64),
        train_samples['item_id'].values.astype(np.int64),
        train_samples['label'].values.astype(np.float32),
    )
    train_loader = DataLoader(train_ds, batch_size=256, shuffle=True, num_workers=0)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(ncf_model.parameters(), lr=5e-4, weight_decay=1e-5)
    for epoch in range(15):
        ncf_model.train()
        total_loss, n_batches = 0.0, 0
        for u, i, y in tqdm(train_loader, desc=f'NCF ep {epoch+1}/15'):
            u, i, y = u.to(device), i.to(device), y.to(device)
            preds = ncf_model(u, i)
            loss = criterion(preds, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            n_batches += 1
    torch.save(ncf_model.state_dict(), NCF_CHECKPOINT)
    print(f'NCF сохранён в {NCF_CHECKPOINT}')

ncf_model.eval()
ncf_candidates = {}
items_t = torch.arange(n_items, device=device, dtype=torch.long)
with torch.no_grad():
    for uid in tqdm(sample_users, desc='NCF top-50'):
        users_t = torch.full_like(items_t, fill_value=int(uid))
        scores = ncf_model(users_t, items_t).cpu().numpy()
        scores = np.nan_to_num(scores, nan=-np.inf)
        bought = user_train_items_cache.get(int(uid), set())
        if bought:
            scores[list(bought)] = -np.inf
        top_idx = np.argpartition(-scores, K_CANDIDATES)[:K_CANDIDATES]
        top_idx = top_idx[np.argsort(-scores[top_idx])]
        ncf_candidates[int(uid)] = [(int(i), float(scores[i])) for i in top_idx]

print(f'NCF candidates готовы для {len(ncf_candidates)} пользователей')


In [ ]:
try:
    del ncf_model
except NameError:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

_eot_id = tokenizer.convert_tokens_to_ids('<|eot_id|>')
LLM_TERMINATORS = sorted({
    int(t) for t in [tokenizer.eos_token_id, _eot_id]
    if isinstance(t, int) and t is not None and t >= 0
})
print(f'Terminators: {LLM_TERMINATORS}')

_llm_kwargs = dict(
    model=LLM_MODEL,
    max_model_len=8192,
    gpu_memory_utilization=0.9,
    enforce_eager=False,
    trust_remote_code=False,
)
if LLM_QUANT is None:
    _llm_kwargs['dtype'] = 'bfloat16'
else:
    _llm_kwargs['quantization'] = LLM_QUANT
    _llm_kwargs['dtype'] = 'auto'
llm = LLM(**_llm_kwargs)
print('vLLM loaded')


In [ ]:
SYSTEM_PROMPT_RERANK_V2 = (
    'Ты — ассистент-рекомендатель детского интернет-магазина. '
    'Тебе даны: профиль пользователя, его история покупок и пронумерованный список из {n} '
    'товаров-кандидатов. Каждый кандидат описан названием, категорией, типом и средней ценой. '
    'Твоя задача — переупорядочить кандидатов от самого вероятного к покупке до наименее, '
    'опираясь на возраст ребёнка (выводимый из истории), бренды, категории, ценовой сегмент, '
    'географию доставки. Используй ТОЛЬКО номера из списка, не повторяй и не пропускай ни один. '
    'Отвечай СТРОГО в формате JSON без пояснений: '
    '{{"ranked": [номер1, номер2, ..., номер{n}]}}.'
)


def format_user_profile(uid: int) -> str:
    feat = user_feat_dict.get(int(uid), {})
    if not feat:
        return 'Профиль пользователя: нет данных.'
    top_cats = feat.get('user_top_cats') or []
    top_cats_str = ', '.join(top_cats) if top_cats else 'нет данных'
    avg_price = feat.get('user_avg_price', 0) or 0
    return (
        'Профиль пользователя:\n'
        f'- Топ-категории: {top_cats_str}\n'
        f'- Средний чек: {avg_price:.0f} руб\n'
        f'- География: {feat.get("user_geo", "неизвестно")}\n'
        f'- Кол-во заказов: {int(feat.get("user_n_purchases", 0))}, '
        f'уникальных товаров: {int(feat.get("user_n_unique_items", 0))}\n'
        f'- Предпочитаемая доставка: {feat.get("user_delivery_method", "неизвестно")}'
    )


def format_candidate_line(idx: int, iid: int) -> str:
    name = item_name_map.loc[iid]
    feat = item_feat_dict.get(int(iid), {})
    cat2 = feat.get('Группа2', 'н/д')
    cat3 = feat.get('Группа3', 'н/д')
    typ = feat.get('Тип', 'н/д')
    price = feat.get('item_avg_price', 0) or 0
    return (
        f'{idx + 1}. {name} | категория: {cat2} > {cat3} | '
        f'тип: {typ} | средняя цена: {price:.0f} руб'
    )


def build_rerank_prompt_v2(user_id: int, candidate_item_ids):
    profile = format_user_profile(user_id)
    hist = build_user_history(user_id, max_items=MAX_HISTORY_ITEMS)
    cand_lines = '\n'.join(format_candidate_line(idx, iid) for idx, iid in enumerate(candidate_item_ids))
    n = len(candidate_item_ids)
    user_prompt = (
        f'{profile}\n\n'
        f'История покупок (последние {MAX_HISTORY_ITEMS}):\n{hist}\n\n'
        f'Кандидаты для переупорядочивания (всего {n}):\n{cand_lines}\n\n'
        f'Верни JSON с полем "ranked" — массивом из {n} разных номеров (1..{n}) '
        f'в порядке от самого подходящего к наименее. Только JSON, без поясняющего текста.'
    )
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT_RERANK_V2.format(n=n)},
        {'role': 'user', 'content': user_prompt},
    ]
    return tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)


# Демо для самопроверки
demo_uid = sample_users[0]
demo_cands = [iid for iid, _ in als_candidates[demo_uid][:5]]
demo_prompt = build_rerank_prompt_v2(demo_uid, demo_cands)
print(demo_prompt[:1500])
print('...')


In [ ]:
def parse_ranked_indices(raw_text: str, n_candidates: int):
    arr = None
    try:
        obj = json.loads(raw_text)
        if isinstance(obj, dict):
            for key in ('ranked', 'order', 'indices', 'rank'):
                if key in obj and isinstance(obj[key], list):
                    arr = obj[key]
                    break
            if arr is None:
                for v in obj.values():
                    if isinstance(v, list):
                        arr = v
                        break
        elif isinstance(obj, list):
            arr = obj
    except Exception:
        m = re.search(r'\[[\s\S]*?\]', raw_text)
        if m:
            try:
                arr = json.loads(m.group(0))
            except Exception:
                arr = None

    if arr is None:
        return None

    seen = set()
    result = []
    for x in arr:
        try:
            i = int(x) - 1
        except Exception:
            continue
        if 0 <= i < n_candidates and i not in seen:
            seen.add(i)
            result.append(i)

    if not result:
        return None

    for i in range(n_candidates):
        if i not in seen:
            result.append(i)
    return result


In [ ]:
rerank_v2_cache = {
    'als_raw': {}, 'ncf_raw': {},
}
if LLM_RERANK_V2_CACHE.exists():
    with open(LLM_RERANK_V2_CACHE, 'r', encoding='utf-8') as f:
        loaded = json.load(f)
    for k in rerank_v2_cache:
        rerank_v2_cache[k] = {int(uid): v for uid, v in loaded.get(k, {}).items()}
    n_raw = sum(len(v) for v in rerank_v2_cache.values())
    print(f'Cache v2: восстановлено {n_raw} сырых ответов')

sources = [
    ('als',     als_candidates,  'als_raw'),
    ('ncf',     ncf_candidates,  'ncf_raw'),
]

tasks = []
prompts = []
for src_name, cand_dict, raw_key in sources:
    for uid in sample_users:
        uid = int(uid)
        if uid in rerank_v2_cache[raw_key]:
            continue
        cand_ids = [iid for iid, _ in cand_dict[uid]]
        tasks.append((raw_key, uid, cand_ids))
        prompts.append(build_rerank_prompt_v2(uid, cand_ids))

print(f'К обработке промптов: {len(prompts)}')

if prompts:
    sample_lens = [len(tokenizer.encode(p)) for p in prompts[:50]]
    p50, p95, p99 = np.percentile(sample_lens, [50, 95, 99])
    print(f'Длина промпта (по первым 50): p50={p50:.0f}, p95={p95:.0f}, p99={p99:.0f} токенов')

    sampling_params = SamplingParams(
        temperature=0.0,
        max_tokens=400,
        stop_token_ids=LLM_TERMINATORS,
    )

    t0 = time.time()
    outputs = llm.generate(prompts, sampling_params)
    elapsed = time.time() - t0
    total_gen = sum(len(o.outputs[0].token_ids) for o in outputs)
    print(f'\nГенерация {len(prompts)} промптов за {elapsed:.1f}s '
          f'({elapsed / len(prompts):.2f}s/промпт, {total_gen / elapsed:.0f} tok/sec)')

    for (raw_key, uid, cand_ids), out in zip(tasks, outputs):
        rerank_v2_cache[raw_key][uid] = out.outputs[0].text

with open(LLM_RERANK_V2_CACHE, 'w', encoding='utf-8') as f:
    json.dump(
        {k: {str(uid): v for uid, v in d.items()} for k, d in rerank_v2_cache.items()},
        f, ensure_ascii=False
    )
print(f'Кэш v2 сохранён в {LLM_RERANK_V2_CACHE}')

# Fallback rate
def fallback_count(raw_dict):
    n_total, n_fb = 0, 0
    for uid in sample_users:
        n_total += 1
        if parse_ranked_indices(raw_dict.get(int(uid), ''), K_CANDIDATES) is None:
            n_fb += 1
    return n_fb, n_total

print('\nFallback rate (невалидный JSON → исходный порядок):')
for name, key in [('ALS', 'als_raw'), ('NCF', 'ncf_raw')]:
    fb, tot = fallback_count(rerank_v2_cache[key])
    print(f'  {name}: {fb}/{tot} ({fb/tot*100:.1f}%)')


In [ ]:
def llm_perm_or_identity(raw_text: str, n: int):
    perm = parse_ranked_indices(raw_text, n)
    if perm is None:
        return list(range(n))  # фолбэк: исходный порядок
    return perm


def rrf_recommend(uid: int, cand_dict: dict, raw_dict: dict, alpha: float, k_rrf: int = RRF_K):
    cand_ids = [iid for iid, _ in cand_dict[int(uid)]]
    n = len(cand_ids)
    perm = llm_perm_or_identity(raw_dict.get(int(uid), ''), n)
    rank_als = {cand_ids[i]: i + 1 for i in range(n)}
    rank_llm = {cand_ids[perm_i]: i + 1 for i, perm_i in enumerate(perm)}
    scores = {iid: 1.0 / (k_rrf + rank_als[iid]) + alpha * 1.0 / (k_rrf + rank_llm[iid])
              for iid in cand_ids}
    ordered = sorted(cand_ids, key=lambda x: -scores[x])
    return ordered[:FINAL_K]


def llm_only_recommend(uid: int, cand_dict: dict, raw_dict: dict):
    cand_ids = [iid for iid, _ in cand_dict[int(uid)]]
    n = len(cand_ids)
    perm = llm_perm_or_identity(raw_dict.get(int(uid), ''), n)
    return [cand_ids[perm_i] for perm_i in perm][:FINAL_K]


def build_recs_rrf(cand_dict: dict, raw_dict: dict, alpha: float):
    return {int(uid): rrf_recommend(int(uid), cand_dict, raw_dict, alpha)
            for uid in sample_users}


def build_recs_llm_only(cand_dict: dict, raw_dict: dict):
    return {int(uid): llm_only_recommend(int(uid), cand_dict, raw_dict)
            for uid in sample_users}


In [ ]:
K_VALUES = [5, 10, 20]


def precision_at_k(recommended, relevant, k):
    rec_k = set(recommended[:k])
    return len(rec_k & set(relevant)) / len(rec_k) if rec_k else 0.0


def recall_at_k(recommended, relevant, k):
    rec_k = set(recommended[:k])
    return len(rec_k & set(relevant)) / len(set(relevant)) if relevant else 0.0


def map_at_k(recommended, relevant, k):
    relevant = set(relevant)
    if not relevant:
        return 0.0
    score, hits = 0.0, 0.0
    for i, item in enumerate(recommended[:k]):
        if item in relevant:
            hits += 1
            score += hits / (i + 1)
    return score / min(len(relevant), k)


def ndcg_at_k(recommended, relevant, k):
    relevant = set(relevant)
    if not relevant:
        return 0.0
    dcg = sum(1.0 / np.log2(i + 2) for i, item in enumerate(recommended[:k]) if item in relevant)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(min(len(relevant), k)))
    return dcg / idcg if idcg > 0 else 0.0


def aggregate(recs_by_user, users):
    out = {k: {m: [] for m in ['precision', 'recall', 'map', 'ndcg']} for k in K_VALUES}
    for u in users:
        u = int(u)
        rel = test_user_items.get(u, [])
        if not rel:
            continue
        rec = recs_by_user.get(u, [])
        for k in K_VALUES:
            out[k]['precision'].append(precision_at_k(rec, rel, k))
            out[k]['recall'].append(recall_at_k(rec, rel, k))
            out[k]['map'].append(map_at_k(rec, rel, k))
            out[k]['ndcg'].append(ndcg_at_k(rec, rel, k))
    return {k: {m: float(np.mean(v)) if v else 0.0 for m, v in metrics.items()}
            for k, metrics in out.items()}


In [ ]:
als_recs_base = {int(uid): [iid for iid, _ in als_candidates[int(uid)][:FINAL_K]] for uid in sample_users}
ncf_recs_base = {int(uid): [iid for iid, _ in ncf_candidates[int(uid)][:FINAL_K]] for uid in sample_users}

sanity_als = build_recs_rrf(als_candidates, rerank_v2_cache['als_raw'], alpha=0.0)
match = all(sanity_als[u] == als_recs_base[u] for u in sample_users)
print(f'Sanity (alpha=0 vs base): {"OK" if match else "FAIL"}')
if not match:
    diffs = [u for u in sample_users if sanity_als[u] != als_recs_base[u]][:3]
    print(f'  отличия для: {diffs}')

sources_for_eval = [
    ('ALS', als_candidates, 'als_raw', als_recs_base),
    ('NCF', ncf_candidates, 'ncf_raw', ncf_recs_base),
]

all_results = {}
for src_name, cand_dict, raw_key, base_recs in sources_for_eval:
    all_results[src_name] = aggregate(base_recs, sample_users)
    all_results[f'{src_name} + LLM only'] = aggregate(
        build_recs_llm_only(cand_dict, rerank_v2_cache[raw_key]), sample_users
    )
    for alpha in ALPHA_SWEEP:
        recs = build_recs_rrf(cand_dict, rerank_v2_cache[raw_key], alpha)
        label = f'{src_name} + LLM rerank v2 (RRF α={alpha})'
        all_results[label] = aggregate(recs, sample_users)

print(f'\nВсего конфигураций: {len(all_results)}')


In [ ]:
rows = []
for name, res in all_results.items():
    row = {'model': name}
    for k in K_VALUES:
        for m in ('precision', 'recall', 'map', 'ndcg'):
            row[f'{m}@{k}'] = res[k][m]
    rows.append(row)
comp_df = pd.DataFrame(rows).set_index('model')
print('Сводная таблица метрик:')
display(comp_df.style.format('{:.4f}'))


In [ ]:

delta_rows = []
for src_name in ['ALS', 'NCF']:
    base_res = all_results[src_name]
    for label_suffix in ['LLM only'] + [f'LLM rerank v2 (RRF α={a})' for a in ALPHA_SWEEP]:
        key = f'{src_name} + {label_suffix}'
        if key not in all_results:
            continue
        rer_res = all_results[key]
        delta = {
            f'{m}@{k}': (rer_res[k][m] - base_res[k][m]) / (base_res[k][m] + 1e-12) * 100
            for k in K_VALUES for m in ('precision', 'recall', 'map', 'ndcg')
        }
        delta_rows.append({'pair': f'{src_name} → {label_suffix}', **delta})

delta_df = pd.DataFrame(delta_rows).set_index('pair')
print('\nПрирост от LLM rerank v2 (% относительно базы):')
display(delta_df.style.format('{:+.1f}%'))


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
metrics_list = ['precision', 'recall', 'map', 'ndcg']
colors = {
    'ALS': 'tab:blue',
    'NCF': 'tab:orange',
}
linestyles = {
    0.0: ':', 0.1: '-.', 0.3: '--', 0.5: '-', 1.0: (0, (3, 1, 1, 1)),
}

for ax, metric in zip(axes.flat, metrics_list):
    for src_name in ['ALS', 'NCF']:
        base_res = all_results[src_name]
        ax.plot(K_VALUES, [base_res[k][metric] for k in K_VALUES],
                marker='o', linestyle='-', linewidth=2,
                color=colors[src_name], label=f'{src_name} (base)')
        llm_only_res = all_results.get(f'{src_name} + LLM only')
        if llm_only_res:
            ax.plot(K_VALUES, [llm_only_res[k][metric] for k in K_VALUES],
                    marker='D', linestyle=(0, (1, 1)), linewidth=1.5,
                    color=colors[src_name], alpha=0.9,
                    label=f'{src_name} LLM only')
        for alpha in ALPHA_SWEEP:
            label = f'{src_name} + LLM rerank v2 (RRF α={alpha})'
            res = all_results[label]
            ax.plot(K_VALUES, [res[k][metric] for k in K_VALUES],
                    marker='s', linestyle=linestyles[alpha], linewidth=1,
                    color=colors[src_name], alpha=0.7,
                    label=f'{src_name} α={alpha}')
    ax.set_xlabel('K')
    ax.set_ylabel(f'{metric.upper()}@K')
    ax.set_title(f'{metric.upper()}@K')
    ax.legend(fontsize=7, ncol=2)
    ax.grid(True, alpha=0.3)

plt.suptitle(f'LLM rerank v2: RRF alpha sweep (n={len(sample_users)})', fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
uid = int(sample_users[0])
print(f'=== uid={uid} ===\n')
print(format_user_profile(uid))
print('\nИстория (последние 10):')
for date, name in user_history_cache.get(uid, [])[-10:]:
    print(f'  {date}: {name[:80]}')

print('\n--- ALS top-5 (base) ---')
for iid, sc in als_candidates[uid][:5]:
    feat = item_feat_dict.get(iid, {})
    print(f'  {iid}: {item_name_map.loc[iid][:60]}  | {feat.get("Группа2", "?")} '
          f'| {feat.get("item_avg_price", 0):.0f}р | als_score={sc:.3f}')

print('\n--- LLM raw для ALS (первые 250 симв.) ---')
print(rerank_v2_cache['als_raw'].get(uid, '')[:250])

print('\n--- ALS top-5 после RRF α=0.3 ---')
recs_03 = rrf_recommend(uid, als_candidates, rerank_v2_cache['als_raw'], alpha=0.3)
for iid in recs_03[:5]:
    print(f'  {iid}: {item_name_map.loc[iid][:60]}')

print('\n--- ALS top-5 после RRF α=1.0 ---')
recs_10 = rrf_recommend(uid, als_candidates, rerank_v2_cache['als_raw'], alpha=1.0)
for iid in recs_10[:5]:
    print(f'  {iid}: {item_name_map.loc[iid][:60]}')

print('\n--- Test items (первые 10) ---')
for iid in test_user_items.get(uid, [])[:10]:
    print(f'  {iid}: {item_name_map.loc[iid][:60]}')
